# Positive Definiteness Check

Chapter 7: Symmetric Matrix · *Mathematics for AI and Machine Learning* by Fuheng Wu

Track eigenvalue and principal-minor conditions for SPD behavior.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-7/positive-definite/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's default symmetric matrix
a11, a12, a22 = 1.5, 0.3, 1.9
A = torch.tensor([[a11, a12],
                  [a12, a22]], dtype=torch.float64)


def spd_report(A):
    evals = torch.linalg.eigvalsh(A)
    minors = (A[0, 0].item(), torch.linalg.det(A).item())
    try:
        torch.linalg.cholesky(A)
        chol = "succeeds"
    except RuntimeError:
        chol = "fails"
    return evals, minors, chol


# ── Three equivalent tests for SPD ──────────────────────────────────────────
evals, minors, chol = spd_report(A)
print("eigenvalues                =", evals.tolist(), "-> all > 0:", bool((evals > 0).all()))
print("leading minors (Sylvester) =", minors, "-> all > 0:", all(m > 0 for m in minors))
print("Cholesky A = L L^T          :", chol)

# x^T A x > 0 for many random nonzero x
torch.manual_seed(0)
X = torch.randn(10000, 2, dtype=torch.float64)
energies = (X @ A * X).sum(dim=1)
print("min of x^T A x over 10000 random x =", energies.min().item())

# ── Push a12 up: SPD stops once a12^2 >= a11 * a22 ──────────────────────────
print("\nboundary: |a12| < sqrt(a11 a22) =", (a11 * a22) ** 0.5)
for b in (0.3, 1.5, 1.68, 1.8):
    B = torch.tensor([[a11, b], [b, a22]], dtype=torch.float64)
    ev, mi, ch = spd_report(B)
    print(f"a12={b:4.2f}: lambda_min={ev[0].item():+.4f}  det={mi[1]:+.4f}  Cholesky {ch}")

eigenvalues                = [1.3394448724536012, 2.0605551275463987] -> all > 0: True
leading minors (Sylvester) = (1.5, 2.76) -> all > 0: True
Cholesky A = L L^T          : succeeds
min of x^T A x over 10000 random x = 0.0004625901430407867

boundary: |a12| < sqrt(a11 a22) = 1.688194301613413
a12=0.30: lambda_min=+1.3394  det=+2.7600  Cholesky succeeds
a12=1.50: lambda_min=+0.1867  det=+0.6000  Cholesky succeeds
a12=1.68: lambda_min=+0.0081  det=+0.0276  Cholesky succeeds
a12=1.80: lambda_min=-0.1111  det=-0.3900  Cholesky fails
